# Labelled examples

This exploratory notebook serves for the creation of labelled examples

In [42]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from src.text_processing_functions import *
from src.LLM_functions import *
import copy as cp

from src.data import *



In [43]:
#Load data
file_path = DATA_IN_JSONS +'filtered_report_types_nat_hazards_summary-header.json'#'nathaz_ifrc_reports_info_processed.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    filtered_reports = json.load(json_file)
filtered_reports = pd.DataFrame(filtered_reports)

## Report labelling

Choose a hazard directory. In the case below we use : \
hazard_all_subtype_emdat = {
“drought”, 
“forest fire”, “land fire”, 
“ground movement”, “tsunami”, 
“avalanche”, “landslide”, “rockfall”, “sudden subsidence”, “mudslide", 
“ash fall”, “lava flow”, “pyroclastic flow”, “lahar”, 
“coastal flood”, “flash flood”, “riverine flood”, “ice jam flood”,
“rogue wave”, “seiche”, 
”coldwave”, “heatwave”, “severe winter conditions”, 
“derecho”, “hail”, “lightning/thunderstorm”, “sand/dust storm”,  “winter storm/blizzard”, “storm surge”, “tornado”, “extra-tropical storm”, “tropical cyclone”
}

The flood subtype being hard to differentiate, we will assign hazard to "flash flood" when the text mention heavy rain, "riverine flood" if nothing specific is mentioned. 


Choose a hazard dict : 
hazard_subtype_emdat = {
'Drought': r"drought.", 

'Wildfire': r"wildfire.|forest fire.|land fire." , 

‘Earthquake’ : r”ground movement.|tsunami.”, 

‘Mass movement’: r"avalanche.|landslide.|rockfall.|sudden subsidence.|mudslide.",

‘Volcanic activity’ : r“ash fall.|lava flow.|pyroclastic flow.|lahar”, 

'Flood': r"(coastal flood.|flash flood.|riverine flood.|ice jam flood.)",

‘Wave action’ : r“rogue wave.|seiche”,

‘Extreme temperature’ : r”coldwave.|heatwave.|severe winter conditions.”, 

‘Storm’ : r”derecho.|hail.|lightning.|winterstorm.|storm surge.|tornado.|winter storm.|extra-tropical storm.|tropical storm.”
}


In [44]:
# select reports to be labelled
appealCode_luca = ["MDRLA009",
"MDRMG020",
"MDRNI012",
"MDRBZ006",
"MDRCN006",
"MDRBD022",
"MDRYE011",
"MDRS2001",
"MDRIQ014",
"MDRGN015",
"MDRSV012",
"MDRMY003",
"MDRBD015"]

reports_to_label_luca = filtered_reports.where(filtered_reports.appealCode.isin(appealCode_luca)).dropna()

#convert dates
reports_to_label_luca.date = pd.to_datetime(reports_to_label_luca.date, dayfirst=True)

#select most recent reports
reports_to_label_luca = reports_to_label_luca.groupby('appealCode').apply(lambda x: x.sort_values('date', ascending=False).head(1))

#check that everything is there
print(f"Number appealCodes: {len(appealCode_luca)}, number reports: {len(reports_to_label_luca)}")

Number appealCodes: 13, number reports: 13


In [45]:
reports_to_label_luca

,,reportName,disasterType,date,reportLink,location,appealCode,appealType,origType,pdfDownloaded,text,disasterTypeReclassified,disasterTypeFlag,naturalHazard,text_processed,sentences,nathaz_text
appealCode,,,,,,,,,,,,,,,,,
MDRBD015,1692,Bangladesh - Cyclone Komen (MDRBD015),Cyclone,2016-03-04,https://adore.ifrc.org/Download.aspx?FileId=12...,Bangladesh,MDRBD015,Operations Update,Operations Update 3,1.0,1 | P a g e \n \n \nEmergency appeal n° MDRBD...,Cyclone,0.0,1.0,1 | P a g e Emergency appeal n° MDRBD015 GLIDE...,[1 | P a g e Emergency appeal n° MDRBD015 GLID...,[The Government district level ‘D-form’ data i...
MDRBD022,814,Bangladesh - Monsoon Floods (MDRBD022),[Flood],2020-12-05,https://adore.ifrc.org/Download.aspx?FileId=36...,Bangladesh,MDRBD022,Operations Update,Operations Update 4,1.0,\nIFRC Internal \n \nEmergency Appeal n° MDRB...,Flood,0.0,1.0,IFRC Internal Emergency Appeal n° MDRBD022 GLI...,[IFRC Internal Emergency Appeal n° MDRBD022 GL...,[SITUATION ANALYSIS Description of the disaste...
MDRBZ006,666,Belize - Hurricane Eta (MDRBZ006),All other disaster and emergencies,2021-08-19,https://adore.ifrc.org/Download.aspx?FileId=43...,Belize,MDRBZ006,DREF Operation Final Report,DREF Final Report,1.0,\n \nPublic \n \nDREF Operation n° MDRBZ006 \...,Cyclone,0.0,1.0,Public DREF Operation n° MDRBZ006 GLIDE n° TC-...,[Public DREF Operation n° MDRBZ006 GLIDE n° TC...,"[The remaining balance of 131,747 CHF will be ..."
MDRCN006,1150,China - Floods (MDRCN006),Flood,2019-03-14,https://adore.ifrc.org/Download.aspx?FileId=23...,China,MDRCN006,DREF Operation Final Report,Final report with updated financial report,1.0,\n \nDREF operation \nOperation n° MDRCN006...,Flood,0.0,1.0,DREF operation Operation n° MDRCN006 Date of I...,[DREF operation Operation n° MDRCN006 Date of ...,[SITUATION ANALYSIS Description of the disaste...
MDRGN015,193,Guinea - Floods : Coyah (MDRGN015),Flood,2023-11-30,https://adore.ifrc.org/Download.aspx?FileId=76...,Guinea,MDRGN015,DREF Operation Update,MDRGN015ou1,1.0,Page 1 / 19\nDREF Operational Update\nGuinea F...,Flood,0.0,1.0,Page 1 / 19 DREF Operational Update Guinea Flo...,[Page 1 / 19 DREF Operational Update Guinea Fl...,[Page 1 / 19 DREF Operational Update Guinea Fl...
MDRIQ014,324,Iraq - Flash Floods (MDRIQ014),Pluvial/Flash Flood,2023-05-12,https://adore.ifrc.org/Download.aspx?FileId=67...,Iraq,MDRIQ014,DREF Operation Final Report,MDRIQ014fr,1.0,\n \nPublic \n \nDREF Operation \nOperation n...,Flood,0.0,1.0,Public DREF Operation Operation n° MDRIQ014 Da...,[Public DREF Operation Operation n° MDRIQ014 D...,[The major donors and partners of the Disaster...
MDRLA009,69,Laos - Flood (MDRLA009),Flood,2024-05-31,https://adore.ifrc.org/Download.aspx?FileId=83...,Lao People'S Democratic Republic,MDRLA009,DREF Operation Final Report,MDRLA009fnr,1.0,DREF Final Report\nDREF Laos Flood 2023\nAffec...,Flood,0.0,1.0,DREF Final Report DREF Laos Flood 2023 Affecte...,[DREF Final Report DREF Laos Flood 2023 Affect...,"[(Map: IFRC, IM) Date when the trigger was met..."
MDRMG020,107,Madagascar - Tropical Cyclone Freddy (MDRMG020),Cyclone,2024-04-04,https://adore.ifrc.org/Download.aspx?FileId=82...,Madagascar,MDRMG020,DREF Operation Final Report,MDRMG020dfr,1.0,Page 1 / 18\nDREF Final Report\nMadagascar Tro...,Cyclone,0.0,1.0,Page 1 / 18 DREF Final Report Madagascar Tropi...,[Page 1 / 18 DREF Final Report Madagascar Trop...,[Page 2 / 18 Description of the Event Date of ...
MDRMY003,1384,Malaysia - Floods (MDRMY003),Flood,2017-11-21,https://adore.ifrc.org/Download.aspx?FileId=17...,Malaysia,MDRMY003,DREF Operation Final Report,MDRMY003DREF_FR,1.0,\n \n \nDREF operation n° MDRMY003 \nGlide n°...,Flood,0.0,1.0,DREF operation n° MDRMY003 Glide n° FL-2017-00...,[DREF operation n° MDRMY003 Glide n° FL-2017-0...,[Situation analysis Description of the disaste...


In [46]:
for groupname, group in reports_to_label_luca.groupby("origType"):
    if len(group) > 1:
        print(f"{groupname}: {len(group)}")

In [47]:
reports_bang = reports_to_label_luca.where(reports_to_label_luca.appealCode == "MDRBD015").dropna(how='all')

In [48]:
i=0
print(f"{reports_bang.iloc[i].origType}: {reports_bang.iloc[i].date}")
reports_bang.iloc[i].nathaz_text

Operations Update 3: 2016-03-04 00:00:00


['The Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.',
 'Crops were damaged and shrimp projects flooded.',
 'Due to the impact of the cyclonic storm, heavy to very heavy rainfall triggered in southern Bangladesh widespread flooding.',
 'Consequently the lives and livelihoods of the people of those areas were further worsened.',
 'Emergency appeal operations update Bangladesh: Cyclone Komen 2 | P a g e A Need Assessment Working Group (NAWG) was formed to identify the damage and needs of all these areas affected by Cyclone Komen and subsequent flooding.',
 'This assessment was commissioned by the Humanitarian Coordination Task Team (HCTT) and covered ten districts.',
 'The cumulative effect of the floods coming after Cyclone Komen increased the affected population to 2.6 million people.',
 'The impact of these events was felt 

In [49]:
#i=1
#print(f"{reports_bang.iloc[i].origType}: {reports_bang.iloc[i].date}")
#reports_bang.iloc[i].nathaz_text

In [50]:
#i=2
#print(f"{reports_bang.iloc[i].origType}: {reports_bang.iloc[i].date}")
#reports_bang.iloc[i].nathaz_text

In [51]:
#i=3
#print(f"{reports_bang.iloc[i].origType}: {reports_bang.iloc[i].date}")
#reports_bang.iloc[i].nathaz_text

In [52]:
labelled_reports_dict = {} # dict to store labelled reports
#empty dict structure to store results
labelled_reports_dict["appealCode"]=[
    {"reportDate": None,
     "hazardType": None,
     "hazardSubtypes" : None,
     "country" : None,
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },

]

In [53]:
i=0
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRBD015: 2016-03-04 00:00:00


['The Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.',
 'Crops were damaged and shrimp projects flooded.',
 'Due to the impact of the cyclonic storm, heavy to very heavy rainfall triggered in southern Bangladesh widespread flooding.',
 'Consequently the lives and livelihoods of the people of those areas were further worsened.',
 'Emergency appeal operations update Bangladesh: Cyclone Komen 2 | P a g e A Need Assessment Working Group (NAWG) was formed to identify the damage and needs of all these areas affected by Cyclone Komen and subsequent flooding.',
 'This assessment was commissioned by the Humanitarian Coordination Task Team (HCTT) and covered ten districts.',
 'The cumulative effect of the floods coming after Cyclone Komen increased the affected population to 2.6 million people.',
 'The impact of these events was felt 

In [54]:
labelled_reports_dict['MDRBD015']=[
    {"reportDate": "2015-09-16",
    "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : ["North part",
                 "Central part",
                 "Jamuna River basin",
                 "Brahmaputra River basin",
                 "Assam",
                 "Meghalaya",
                 "West Bengal"],
     "city" : None,
     "locationAnnotation" : ['While BDRCS and IFRC as well as the other humanitarian partners are dealing with the cyclone Komen and flooding in the South Eastern part of Bangladesh, the North and Central part of Bangladesh is experiencing flooding since the last week of August 2015.',
                             'The country is experiencing heavy to very heavy rainfall in the Jamuna and Brahmaputra River basin since last week of August.',
                             'At the same time the upper catchment area of India, namely Assam, Meghalaya and some part of West Bengal also experienced heavy rain.'],

     "startYear" : 2015,
     "startMonth" : 8,
     "startDay" : 20,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    #{"hazardType": "Flood",
    # "hazardSubtypes" : None,
    # "country" : "Bangladesh",
    # "region" : "Jamuna and Brahmaputra River basin",
    # "city" : None,
    # "locationAnnotation" : 'The country is experiencing heavy to very heavy rainfall in the Jamuna and Brahmaputra River basin since last week of August.',
    # "startYear" : 2015,
    # "startMonth" : 8,
    # "startDay" : 20,
    # "endYear" : None,
    # "endMonth" : None,
    # "endDay" : None,
    # "hazardName" : None,
    #},
    #{"hazardType": "Flood",
    # "hazardSubtypes" : None,
    # "country" : "Bangladesh",
    # "region" : "Assam, Meghalaya, West Bengal",
    # "city" : None,
    # "locationAnnotation" : 'At the same time the upper catchment area of India, namely Assam, Meghalaya and some part of West Bengal also experienced heavy rain.',
    # "startYear" : 2015,
    # "startMonth" : 8,
    # "startDay" : 20,
    # "endYear" : None,
    # "endMonth" : None,
    # "endDay" : None,
    # "hazardName" : None,
    #},
    {"reportDate": "2015-09-16",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Bangladesh",
     "region" : ["South Eastern part"],
     "city" : None,
     "locationAnnotation" : ['While BDRCS and IFRC as well as the other humanitarian partners are dealing with the cyclone Komen and flooding in the South Eastern part of Bangladesh, the North and Central part of Bangladesh is experiencing flooding since the last week of August 2015.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Komen",
    }
]

In [55]:
i=1
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRBD022: 2020-12-05 00:00:00


['SITUATION ANALYSIS Description of the disaster The heavy rainfall occurred during July to September 2019 across Bangladesh led to landslides and extreme flooding.',
 'The impacts were much larger in scale than an average annual monsoon flood as it affected millions of lives.',
 'According to the National Needs Assessment Working Group (NAWG), Bangladesh situation report dated 28 July 2019, more than 7.6 million people were affected in 28 districts, over 300,000 people displaced, approximately 600,000 houses damaged, and 114 people dead.',
 'On top of that, according to the media, about 532,000 hectares of crops were destroyed1.',
 'It was also reported that embankments were damaged and inundated.',
 'With the flood prevailing over one fourth areas of the country in July 2019, Government and Non-Government humanitarian agencies supported the affected community immediately after the flood to meet the emergency needs.',
 'In addition, Bangladesh experienced another spell of flood in nor

In [56]:
labelled_reports_dict['MDRBD022']=[
    {"reportDate": "2020-12-05",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster The heavy rainfall occurred during July to September 2019 across Bangladesh led to landslides and extreme flooding."
                             ],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : None,
     "endYear" : 2019,
     "endMonth" : 9,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2020-12-05",
     "hazardType": "Mass Movement",
     "hazardSubtypes" : ["landslide"],
     "country" : "Bangladesh",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster The heavy rainfall occurred during July to September 2019 across Bangladesh led to landslides and extreme flooding."
                             ],
     "startYear" : 2019,
     "startMonth" : 7,
     "startDay" : None,
     "endYear" : 2019,
     "endMonth" : 9,
     "endDay" : None,
     "hazardName" : None,
    },
     {"reportDate": "2020-12-05",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : ["Rajshahi",
                 "Shariatpur",
                 "Kushtia",
                 "Rajbari",
                 "Chapai Nawabganj",
                 "Pabna",
                 "Natore"],
     "city" : None,
     "locationAnnotation" : ["In addition, Bangladesh experienced another spell of flood in north-eastern districts named Rajshahi, Shariatpur, Kushtia, Rajbari, Chapai Nawabganj, Pabna and Natore during the first week of October 2019, which affected some new areas."
                             ],
     "startYear" : 2019,
     "startMonth" : 10,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2020-12-05",
     "hazardType": "Extreme temperature",
     "hazardSubtypes" : ["coldwave"],
     "country" : "Bangladesh",
     "region" : ["Chuadanga",
                 "Dinajpur",
                 "Panchagarh",
                 "Rajshahi",
                 "Pabna",
                 "Naogaon",
                 "Nilphamari",
                 "Jessore",
                 "Bogura",
                 "Lalmonirhat",
                 "Gaibandha",
                 "Kurigram",
                 "Sirajganj",
                 "Tangail",
                 "Jamalpur"],
     "city" : None,
     "locationAnnotation" : ["During December 2019 and January 2020, the country experienced several cold waves over different districts of Chuadanga, Dinajpur, Panchagarh, Rajshahi, Pabna, Naogaon, Nilphamari, Jessore, Bogura, Lalmonirhat, Gaibandha, Kurigram, Sirajganj, Tangail, Jamalpur, etc., disrupting normal life and causing suffering to the people."
                             ],
     "startYear" : 2019,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2020,
     "endMonth" : 1,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2020-12-05",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Bangladesh",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["In 2020, in addition to COVID-19, Cyclone Amphan, weakened from a super cyclone to an ‘extremely severe cyclonic storm’ slammed into the coastal districts of West Bengal, India and then it entered Bangladesh on 20 May 2020 evening with wind speed of 150 kilometers per hour and caused huge destruction in 26 districts across the country.",
                             ],
     "startYear" : 2020,
     "startMonth" : 5,
     "startDay" : 20,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Amphan",
    },
    {"reportDate": "2020-12-05",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "India",
     "region" : ["West Bengal"],
     "city" : None,
     "locationAnnotation" : ["In 2020, in addition to COVID-19, Cyclone Amphan, weakened from a super cyclone to an ‘extremely severe cyclonic storm’ slammed into the coastal districts of West Bengal, India and then it entered Bangladesh on 20 May 2020 evening with wind speed of 150 kilometers per hour and caused huge destruction in 26 districts across the country.",
                             ],
     "startYear" : 2020,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Amphan",
    },
    {"reportDate": "2020-12-05",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["On the other hand, severe floods that struck Bangladesh during the last week of June 2020, driven by heavy monsoon and upstream water, have prolonged and intensified suffering of 5.4 million people in the northern, central and north- eastern part of the country."
                             ],
     "startYear" : 2020,
     "startMonth" : 6,
     "startDay" : 20,
     "endYear" : 2020,
     "endMonth" : 10,
     "endDay" : 1,
     "hazardName" : None,
    },

]

In [57]:
i=2
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRBZ006: 2021-08-19 00:00:00


['The remaining balance of 131,747 CHF will be returned to the Disaster Relief Emergency Fund.',
 'The major donors and partners of the Disaster Relief Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO, Blizzard Entertainment, Mondelez International Foundation, Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the CRRC, would like to extend thanks to all for their generous contributions.',
 '<Click here for the final financial report and here for the contact information.',
 'SITUATION ANALYSIS Description of the disaster Hurricane Eta made landfall on Nicaragua’s shores as a strong Category 4 hurricane on November 4, 2020, causing destruction and excessive rain with a wind speed of 140 mph.',
 'Several Central American countries experienced the negative effects 

In [58]:
pd.to_datetime("25/03/2021", dayfirst=True)

Timestamp('2021-03-25 00:00:00')

In [59]:
labelled_reports_dict['MDRBZ006']=[
    {"reportDate": "2021-03-25",
     "hazardType": 'Storm',
     "hazardSubtypes" : ["tropical storm"],
     "country" : 'Nicaragua',
     "region" : None,
     "city" : None,
     "locationAnnotation" : ['SITUATION ANALYSIS Description of the disaster Hurricane Eta made landfall on Nicaragua’s shores as a strong Category 4 hurricane on 4 November 2020, causing destruction and excessive rain with a wind speed of 140 mph.'],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 4,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : 'Eta',
    },
    {"reportDate": "2021-03-25",
     "hazardType": 'Storm',
     "hazardSubtypes" : ["tropical storm"],
     "country" : 'Belize',
     "region" : None,
     "city" : None,
     "locationAnnotation" : ['Several Central American countries experienced the negative effects of Hurricane Eta, including Belize.'],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : 'Eta',
    },
    {"reportDate": "2021-03-25",
     "hazardType": 'Flood',
     "hazardSubtypes" : None,
     "country" : 'Belize',
     "region" : ["Western District of Cayo",
                 "Belize District",
                 "Mopan river",
                 "Macal river",
                 "Belize river",
                 "Sibun river"
                 "Cayo District",
                 ],
     "city" : ["Belize City",
               "Arenal",
               "Roaring Creek"],
     "locationAnnotation" :  ['Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
                              "More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers."
                              "In the Cayo District, the Macal and Mopan rivers rose more than 8.8 meters, inundating every village from Arenal to Roaring Creek."],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2021-03-25",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Nicaragua",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["Additionally, on 16 November, Hurricane Iota made landfall in Nicaragua, which brought additional rain to Belize and exacerbated the floods in many areas."],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Iota",
    },
    {"reportDate": "2021-03-25",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Belize",
     "region" : None,
     "city" : ["Arenal",
               "Benque",
               "Calla Creek",
               "Bullet Tree Falls",
               "Valley of Peace",
               "Santa Familia",
               "Blackman Eddy",
               "Roaring Creek",
               "La Rivera",
               "Bomba",
               "Maskall",
               "Crooked Tree",
               "May Pen",
               "Rancho Dolores",
               "Freetown Sibun",
               "Lemonal"],
     "locationAnnotation" : ["Additionally, on 16 November, Hurricane Iota made landfall in Nicaragua, which brought additional rain to Belize and exacerbated the floods in many areas.",
                             "Among the hardest-hit communities were Arenal, Benque, Calla Creek, Bullet Tree Falls, Valley of Peace, Santa Familia, Blackman Eddy, Roaring Creek, La Rivera, Bomba, Maskall, Crooked Tree, May Pen, Rancho Dolores, Freetown Sibun, and Lemonal."],
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 16,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Iota",
    },
    #{"hazardType": 'Flood',
    # "hazardSubtypes" : None,
    # "country" : 'Belize',
    # "region" : "Belize District",
    # "city" : None,
    # "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
    # "startYear" : 2020,
    # "startMonth" : 11,
    # "startDay" : 3,
    # "endYear" : None,
    # "endMonth" : None,
    # "endDay" : None,
    # "hazardName" : None,
    #},
    #{"hazardType": 'Flood',
    # "hazardSubtypes" : None,
    # "country" : 'Belize',
    # "region" : "Belize District",
    # "city" : "Belize City",
    # "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
    # "startYear" : 2020,
    # "startMonth" : 11,
    # "startDay" : 3,
    # "endYear" : None,
    # "endMonth" : None,
    # "endDay" : None,
    # "hazardName" : None,
    #},
]

In [60]:
i=3
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRCN006: 2019-03-14 00:00:00


['SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.',
 'In some areas of North Central Sichuan, there were heavy rainstorms and torrential rains for four consecutive days.',
 'These were also compounded by the effects of two weather systems in the area; Typhoon Prapiroon, and Typhoon Maria.',
 'According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died; 222,000 had taken emergency resettlement; 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan (that includes 15 cities and 70 counties); more than 900 houses collapsed, and 29,000 houses damaged.',
 'A total of 36,900 hectares of crops were also affected by the flood.',
 'The direct economic loss was estimated to be over 5.3 billion Yuan (approximately CHF 792 million).',
 'Gansu province was hi

In [61]:
labelled_reports_dict["MDRCN006"]=[
    {"reportDate": "2019-03-14",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "China",
     "region" : ["Sichuan",
                 "southeast region of Gansu Province",
                 "North Central Sichuan",
                 "Gansu province",
                 ],
     "city" : ["Deyang",
               "Mianyang",
               "Guangyuan",
               "Tianshui",
               "Zhangye",
               "Pingliang"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster Heavy and continuous rainfall on 7 July 2018 caused flooding in most parts of Sichuan and the southeast region of Gansu Province.",
                             "In some areas of North Central Sichuan, there were heavy rainstorms and torrential rains for four consecutive days.",
                             "According to reports from National Disaster Reduction Commission, as of 13 July 2018, the floods affected 1,381,000 people, where 3 persons died; 222,000 had taken emergency resettlement; 22,000 needed emergency relief in Sichuan prefectures of Deyang, Mianyang, Guangyuan (that includes 15 cities and 70 counties); more than 900 houses collapsed, and 29,000 houses damaged.",
                             "Gansu province was hit even harder, according to the Ministry of Emergency Management.",
                             "The area of Tianshui, Zhangye, Pingliang (including 10 cities and 46 counties) were flooded, and affected 1,519,000 people where 12 died; 4 missing; and 30,000 were evacuated.",
                             "The flooding season was rightly anticipated to continue until the end of August 2018 and more rain fall events were registered."],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 7,
     "endYear" : 2018,
     "endMonth" : 8,
     "endDay" : 31,
     "hazardName" : None,
    },
     {"reportDate": "2019-03-14",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "China",
     "region" : ["Sichuan",
                 "southeast region of Gansu Province",
                 "North Central Sichuan"],
     "city" : None,
     "locationAnnotation" : ['These were also compounded by the effects of two weather systems in the area; Typhoon Prapiroon, and Typhoon Maria.',
                             ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Typhoon Prapiroon",
    },
     {"reportDate": "2019-03-14",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "China",
     "region" : ["Sichuan",
                 "southeast region of Gansu Province",
                 "North Central Sichuan"],
     "city" : None,
     "locationAnnotation" : ['These were also compounded by the effects of two weather systems in the area; Typhoon Prapiroon, and Typhoon Maria.',
                             ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Typhoon Maria",
    },
    {"reportDate": "2019-03-14",
     "hazardType": "Storm",
     "hazardSubtypes" : None,
     "country" : "China",
     "region" : ["Southeast Gansu"],
     "city" : None,
     "locationAnnotation" : ['A heavy rainstorm occurred in Southeast Gansu from 10 to 11 July 2018.',
                             ],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 10,
     "endYear" : 2018,
     "endMonth" : 7,
     "endDay" : 11,
     "hazardName" : None,
    },

]

In [62]:
i=4
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRGN015: 2023-11-30 00:00:00


['Page 1 / 19 DREF Operational Update Guinea Floods Appeal: MDRGN015 Total DREF Allocation: - Crisis Category: Yellow Hazard: Flood Glide Number: FL-2023-000158-GIN People Affected: 24,135 people People Targeted: 14,350 people Event Onset: Sudden Operation Start Date: 2023-08-23 New Operational End Date: 2024-01-31 Total Operating Timeframe: 5 months Additional Allocation Requested: - Targeted Areas: Kindia Page 2 / 19 Description of the Event What happened, where and when?',
 'Guinea has been experiencing persistent torrential rains since the beginning of August 2023.',
 'The highest recorded incidents were on Sunday 6 August 2022 in Coyah, and on Friday 11 August 2023 in Conakry and Siguiri, with rains causing associated impacts, including flooding in low-lying areas as well as the overflow of rivers.',
 'Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.',
 'Different 

In [63]:
labelled_reports_dict["MDRGN015"]=[
    {"reportDate": "2023-11-30",
     "hazardType": "Flood",
     "hazardSubtypes" : ["riverine flood", "flash flood"],
     "country" : "Guinea",
     "region" : ["Kindia",
                 ],
     "city" : ["Coyah",
               "Conakry",
               "Siguiri"],
     "locationAnnotation" : ["Page 1 / 19 DREF Operational Update Guinea Floods Appeal: MDRGN015 Total DREF Allocation: - Crisis Category: Yellow Hazard: Flood Glide Number: FL-2023-000158-GIN People Affected: 24,135 people People Targeted: 14,350 people Event Onset: Sudden Operation Start Date: 2023-08-23 New Operational End Date: 2024-01-31 Total Operating Timeframe: 5 months Additional Allocation Requested: - Targeted Areas: Kindia Page 2 / 19 Description of the Event What happened, where and when?",
                             "The highest recorded incidents were on Sunday 6 August 2022 in Coyah, and on Friday 11 August 2023 in Conakry and Siguiri, with rains causing associated impacts, including flooding in low-lying areas as well as the overflow of rivers.",
                             "Major roads in Conakry and Siguiri were rendered impassable due to the flood waters, heavily constraining vehicles, and pedestrians having to find alternative routes.",],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : 1,
     "endYear" : 2023,
     "endMonth" : 8,
     "endDay" : 11,
     "hazardName" : None,
    },

]

In [64]:
i=5
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRIQ014: 2023-05-12 00:00:00


['The major donors and partners of the Disaster Relief Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, the Republic of Korea, Spain, Sweden, and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the Iraqi Red Crescent Society, would like to extend thanks to all for their generous contributions.',
 'SITUATION ANALYSIS Description of the disaster Iraq is at risk of multiple disasters ranging from natural phenomena such as drought, sandstorms, heatwaves, and floods, to man-made ones.',
 'After one of the driest years in decades, heavy rains slammed Iraq’s northern Kurdish region on 17 December 2021.',
 'The overnight rainfall caused a flash flood in Erbil, the region’s capital, and the Kirkuk governorate in northern Iraq.',
 'Destruc

In [65]:
#empty dict structure to store results
labelled_reports_dict["MDRIQ014"]=[
    {"reportDate": "2023-05-12",
     "hazardType": "Flood",
     "hazardSubtypes" : ["flash flood"],
     "country" : "Iraq",
     "region" : ["northern Kurdish",
                 "Erbil",
                 "Kirkuk",
                 "northern Iraq"],
     "city" : ["Daratu",
               "Qushtapa",
               "Shamamk",
               "Zhyan",
               "Roshinbiri",
               "Bahrka"],
     "locationAnnotation" : ["After one of the driest years in decades, heavy rains slammed Iraq’s northern Kurdish region on 17 December 2021.",
                             'The overnight rainfall caused a flash flood in Erbil, the region’s capital, and the Kirkuk governorate in northern Iraq.',
                              "In the early hours of the morning, muddy water inundated people’s homes in Erbil's Daratu, Qushtapa, Shamamk, Zhyan, Roshinbiri, and Bahrka neighborhoods, forcing inhabitants out of their houses.",
],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"reportDate": "2023-05-12",
     "hazardType": "Drought",
     "hazardSubtypes" : ["drought"],
     "country" : "Iraq",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["After one of the driest years in decades, heavy rains slammed Iraq’s northern Kurdish region on 17 December 2021.",
                             ],
     "startYear" : 2021,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]

In [66]:
i=6
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRLA009: 2024-05-31 00:00:00


['(Map: IFRC, IM) Date when the trigger was met 09-08-2023 What happened, where and when?',
 'During August 2023, persistent heavy rain led to extensive flooding across the central and southern regions of Laos.',
 'The flooding caused damages to numerous farms and houses, affecting thousands of people in the inundated areas.',
 'In the report released on 21 August 2023, the National Disaster Management Committee (NDMC), under the Ministry of Labour and Social Welfare (MOLSW), mentioned that 12 provinces were affected by the floods, including Vientiane Capital, Bokeo, Houaphan, Luang Prabang, Xaignabouli, Xiangkhouang, Vientiane, Bolikhamxai, Khammouan, Savannakhet, Champasak and Xaixomboun.',
 'The impact of the flooding was substantial, with a geographical scope that covered 550 villages across 50 districts within the 12 provinces.',
 'The agriculture sector was heavily affected by the floods, with massive damages to crops, cropland and fishponds, which put households in crisis as the

In [67]:
labelled_reports_dict["MDRLA009"]=[
    {"reportDate": "2024-05-31",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Laos",
     "region" : ["Central region",
                 "Souther region",
                 "Vientiane Capital",
                  "Bokeo",
                  "Houaphan",
                  "Luang Prabang",
                  "Xaignabouli",
                  "Xiangkhouang",
                  "Vientiane",
                  "Bolikhamxai",
                  "Khammouan",
                  "Savannakhet",
                  "Champasak",
                  "Xaixomboun"],
     "city" : None,
     "locationAnnotation" : ["During August 2023, persistent heavy rain led to extensive flooding across the central and southern regions of Laos.",
                             "In the report released on 21 August 2023, the National Disaster Management Committee (NDMC), under the Ministry of Labour and Social Welfare (MOLSW), mentioned that 12 provinces were affected by the floods, including Vientiane Capital, Bokeo, Houaphan, Luang Prabang, Xaignabouli, Xiangkhouang, Vientiane, Bolikhamxai, Khammouan, Savannakhet, Champasak and Xaixomboun.",
                             ],
     "startYear" : 2023,
     "startMonth" : 8,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },

]

In [68]:
i=7
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRMG020: 2024-04-04 00:00:00


['Page 2 / 18 Description of the Event Date of event 2023-02-28 What happened, where and when?',
 'Tropical Cyclone Freddy was one of the longest-lived systems in the Southern Hemisphere.',
 'Freddy formed off the coast of Indonesia in early February 2023 and crossed the southern Indian Ocean, reaching Mauritius and La Réunion.',
 'During its trajectory, Tropical Cyclone Freddy reached the equivalent of a category 5 cyclone and was the first cyclone to exceed this intensity in 2023.',
 'After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).',
 'Tropical Cyclone Freddy weakened from a Category 4 cyclone to a Category 3 cyclone before making landfall, but hit Madagascar with sustained winds of 150km/h.',
 'It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by t

In [69]:
labelled_reports_dict["MDRMG020"]=[
    {"reportDate": "2024-04-04",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Mauritius",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).",
                             ],
     "startYear" : 2023,
     "startMonth" : 2,
     "startDay" : 21,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Freddy",
    },
    {"reportDate": "2024-04-04",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "La Réunion",
     "region" : None,
     "city" : None,
     "locationAnnotation" :["After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).",
                             ],
     "startYear" : 2023,
     "startMonth" : 2,
     "startDay" : 21,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Freddy",
    },
    {"reportDate": "2024-04-04",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["After bringing heavy rains and winds to the islands of Mauritius and La Réunion, Tropical Cyclone Freddy made landfall on the east coast of Madagascar on 21 February 2023 at around 19:00 (local time).",
                             "It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2023,
     "startMonth" : 2,
     "startDay" : 21,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Freddy",
    },
    {"reportDate": "2024-04-04",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2022,
     "startMonth" : 2,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Batsirai",
    },
    {"reportDate": "2024-04-04",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2022,
     "startMonth" : 2,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Eminati",
    },
    {"reportDate": "2024-04-04",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Madagascar",
     "region" : ["Mananjary"],
     "city" : None,
     "locationAnnotation" : ["It made landfall in the north of Mananjary, an area previously hit by two tropical cyclones in February 2022 (Batsirai and Eminati) and by the previous cyclone, Cheneso, a few weeks earlier (January 2023)."],
     "startYear" : 2023,
     "startMonth" : 1,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Cyclone Cheneso",
    },


]

In [70]:
i=8
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRMY003: 2017-11-21 00:00:00


['Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia – Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor – and Sabah in East Malaysia.',
 'More than 23,000 people, mainly from smaller towns and villages in rural areas, had to leave their homes to established relief centres.',
 'The situation improved significantly after the weekend of Lunar New Year (28-29 January), with floodwater receding in several affected districts, allowing families that were in relief centres to return home.',
 'National Agency for Disaster Administration (NADMA) reported that the state of Johor suffered the brunt of rising waters, with more than 8,000 evacuees and a fatality.',
 'More information on the floods can be obtained from Information Bulletin n°1 (issued on 5 January), Information Bulletin n°2 (issued on 27 January) and Information Bulletin

In [71]:
labelled_reports_dict["MDRMY003"]=[
    {"reportDate": "2017-11-21",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Malaysia",
     "region" : ["Johor",
                 "Kelantan",
                 "Pahang",
                 "Perak",
                 "Terengganu",
                 "Malacca",
                 "Selangor",
                 "Sabah"],
     "city" : None,
     "locationAnnotation" : ["Situation analysis Description of the disaster Heavy rains that started in December 2016 continued until late January 2017 in parts of Malaysia, causing flooding in seven states of Peninsular Malaysia – Johor, Kelantan, Pahang, Perak, Terengganu, Malacca and Selangor – and Sabah in East Malaysia.",
                             ],
     "startYear" : 2016,
     "startMonth" : 12,
     "startDay" : None,
     "endYear" : 2017,
     "endMonth" : 1,
     "endDay" : 29,
     "hazardName" : None,
    },

]

In [72]:
i=9
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRNI012: 2023-04-06 00:00:00


['The remaining balance of CHF 30,083 will be returned to the Disaster Response Emergency Fund.',
 'The major donors and partners of the Disaster Response Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO, Blizzard Entertainment, Mondelez International Foundation, Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the Nicaraguan Red Cross, would like to extend thanks to all for their generous contributions.',
 '<Click here for the final financial report and here for the contact information.> A.',
 'SITUATION ANALYSIS Description of the disaster Tropical cyclones are among the natural events that cause the most damage to the population.',
 'Their impact on communities depends on the level of risk to which they are exposed and the level of vulnerability.',
 'Histor

In [73]:
labelled_reports_dict["MDRNI012"]=[
    {"reportDate": "2023-04-06",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Nicaragua",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["Historically, Nicaragua has been impacted by tropical cyclones, the most recent being tropical storm Bonnie in May 2022 and category 1 hurricane Julia in October 2022.",
                             ],
     "startYear" : 2022,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "tropical storm Bonnie",
    },
    {"reportDate": "2023-04-06",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Nicaragua",
     "region" : ["Rivas",
                 "Diriamba",
                 "Managua",
                 "San Rafael del Norte",
                 "Chontales",
                  "Boaco",
                  "Jinotega",
                  "Rivas",
                  "Matagalpa",
                  "León"],
     "city" : None,
     "locationAnnotation" : ["Historically, Nicaragua has been impacted by tropical cyclones, the most recent being tropical storm Bonnie in May 2022 and category 1 hurricane Julia in October 2022.",
                             "The devastating forces of Julia damaged several schools in Rivas, Diriamba, Managua and San Rafael del Norte.",
                             "The areas of Chontales, Boaco, Jinotega, Rivas, Matagalpa and León were among the worst affected regions."],
     "startYear" : 2022,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "hurricane Julia",
    },
    {"reportDate": "2023-04-06",
     "hazardType": "Flood",
     "hazardSubtypes" : ["riverine flood"],
     "country" : "Nicaragua",
     "region" : ["Artiwas river",
                 "Wasminona river",
                 "Okonwas river",
                 "Malacatoya river",
                 "Fonseca river",
                 "Siquia river",
                 "Mico river",
                 "Rama river",
                 "South Atlantic Autonomous Region (RAAS)",
                 "Central Zelaya",
                 "Boaco",
                 "South Caribbean"],
     "city" : ["Rosita",
               "El Rama"],
     "locationAnnotation" : ["Julia caused heavy rainfall, which in turn caused several rivers to overflow, including the Artiwas, Wasminona and Okonwas rivers in the municipality of Rosita, the Malacatoya and Fonseca rivers, Siquia, Mico and Rama, among others, putting the population at risk and damaging social infrastructure such as housing, roads and telecommunications in the South Atlantic Autonomous Region (RAAS), Central Zelaya and Boaco, interruption of electricity and drinking water services, obstruction of roads due to falling trees, among others.",
                             "One of the areas most affected was the municipality of El Rama in the South Caribbean, where three rivers converge: Siquia, Mico and Rama, adding to the more than 70 rivers that overflowed nationwide as a result of the rains, leaving villages under water and entire families lost all their belongings."],
     "startYear" : 2022,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "hurricane Julia",
    },
    {"reportDate": "2023-04-06",
     "hazardType": "Mass Movement",
     "hazardSubtypes" : ["landslide"],
     "country" : "Nicaragua",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2022,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "hurricane Julia",
    },

]

In [74]:
i=10
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRS2001: 2024-08-20 00:00:00


['SITUATION ANALYSIS Description of the crisis Hurricane Beryl emerged as a significant climate event, developing from a monitored tropical wave on June 25, 2024.',
 'The storm rapidly intensified, becoming the first major hurricane of the 2024 Atlantic season and reaching unprecedented strength.',
 'By June 29, 2024, Beryl had attained Category 4 status, setting a record as the earliest Category 4 hurricane in history.',
 'The storm continued to strengthen, reaching Category 5 with maximum sustained winds of 270 km/h by July 1, 2024.',
 'This highlights the increasing severity and unpredictability of hurricanes in the Caribbean, exacerbated by rising sea temperatures.',
 'On July 1, 2024, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.',
 'In Grenada, more than 1,600 people sought refuge in emergency shelters, and over 98% of buildings on Carriacou and Petit Martinique suffered severe damage.',
 'The destruction extended to critical i

In [75]:
labelled_reports_dict["MDRS2001"]=[
    {"reportDate": "2024-08-20",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm", "storm surge"],
     "country" : "Grenada",
     "region" : ["Carriacou",
                 "Petit Martinique"],
     "city" : None,
     "locationAnnotation" : ["On July 1, 2024, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.",
                             "In Grenada, more than 1,600 people sought refuge in emergency shelters, and over 98% of buildings on Carriacou and Petit Martinique suffered severe damage.",
                             ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Hurricane Beryl",
    },
    {"reportDate": "2024-08-20",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm", "storm surge"],
     "country" : "Saint Vincent and the Grenadines",
     "region" : ["Union Island"],
     "city" : None,
     "locationAnnotation" : ["On July 1, 2024, the hurricane made landfall in Grenada and Saint Vincent and the Grenadines as a Category 4 storm.",
                             "Saint Vincent and the Grenadines experienced similar devastation, with 90% of homes on Union Island damaged or destroyed, impacting essential services and leaving many residents without access to healthcare and other services."],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Hurricane Beryl",
    },
    {"reportDate": "2024-08-20",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Jamaica",
     "region" : ["Clarendon",
                 "St. Elizabeth"],
     "city" : ["St. Thomas",
               "Manchester",
               "Westmoreland",
               "Hanover"],
     "locationAnnotation" : ["1 Situation Report 11 - Hurricane Beryl - Grenada and St. Vincent and the Grenadines - 29 July 2024 - PAHO/WHO | Pan American Health Organization Operations Update-2 3 Jamaica experienced widespread damage as Hurricane Beryl once again intensified to Category 5, severely impacting infrastructure and agriculture.",
                             "The hardest-hit areas included Clarendon and St. Elizabeth, with extensive damage reported in St. Thomas, Manchester, Westmoreland, and Hanover.",
                             ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 1,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Hurricane Beryl",
    },


]

In [76]:
i=11
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRSV012: 2019-06-26 00:00:00


['The major donors and partners of the Disaster Relief Emergency Fund (DREF) include the Red Cross Societies and governments of Belgium, Britain, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, New Zealand, Norway, Republic of Korea, Spain, Sweden and Switzerland, as well as DG ECHO and Blizzard Entertainment, Mondelez International Foundation, and Fortive Corporation and other corporate and private donors.',
 'The IFRC, on behalf of the national society, would like to extend thanks to all for their generous contributions.',
 'ECHO and the government of Canada have replenished the DREF in the occasion of this operation.',
 'The total amount spent under this DREF operation was 118,630 CHF.',
 'The remaining balance of 32,041 CHF will be reimbursed to the Disaster Relief Emergency Fund.',
 '< For the Final Financial Report, click here.',
 'For contact information, click here.',
 'Situation analysis Description of the disaster On 6 October, rains began falling over eastern El 

In [77]:
labelled_reports_dict["MDRSV012"]=[
    {"reportDate": "2019-06-26",
     "hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "El Salvador",
     "region" : ["Yucatán channel",
                 "Morazán department",
                 "La Union department",
                 "Sonsonate department",
                 "eastern region"
                 "western El Salvador"],
     "city" : ["El Brazo canton",
               "La Canoa canton",
               "El Tecomatal canton",
               "San Miguel municipality",
               "San Felipe canton",
               "Las Tunas canton",
               "Capitán Lazo canton",
               "Puerto Parada canton",
               "Usulután municipality",
               "Metalío canton"],
     "locationAnnotation" : ["Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No. 14 located near the Honduran Atlantic coast.",
                             "On 7 October, the tropical depression was upgraded to Tropical Storm Michael, which continued moving north over the Yucatán channel toward the System declared a Green Alert for the entire country.",
                             "On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazán department and two in La Union department.",
                             "The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel; the cantons of San Felipe and Las Tunas in La Unión department; the cantons of Capitán Lazo and Puerto Parada in the municipality of Usulután; as well as the canton of Metalío in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador: Floods Photo: Area affected by Hurricane Michael",
                             ],
     "startYear" : 2018,
     "startMonth" : 10,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Storm Michael",
    },
    {"reportDate": "2019-06-26",
     "hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "El Salvador",
     "region" : ["Yucatán channel",
                 "Morazán department",
                 "La Union department",
                 "Sonsonate department",
                 "eastern region"
                 "western El Salvador"],
     "city" : ["El Brazo canton",
               "La Canoa canton",
               "El Tecomatal canton",
               "San Miguel municipality",
               "San Felipe canton",
               "Las Tunas canton",
               "Capitán Lazo canton",
               "Puerto Parada canton",
               "Usulután municipality",
               "Metalío canton"],
     "locationAnnotation" : ["Situation analysis Description of the disaster On 6 October, rains began falling over eastern El Salvador due to the influence of tropical depression No. 14 located near the Honduran Atlantic coast.",
                             "On 7 October, the tropical depression was upgraded to Tropical Storm Michael, which continued moving north over the Yucatán channel toward the System declared a Green Alert for the entire country.",
                             "On 7 October, a Yellow Alert was declared for 29 coastal municipalities, which on 8 October increased to 34 municipalities to include three municipalities in Morazán department and two in La Union department.",
                             "The hardest hit have been the eastern regions, specifically the cantons of El Brazo, La Canoa and El Tecomatal in the municipality of San Miguel; the cantons of San Felipe and Las Tunas in La Unión department; the cantons of Capitán Lazo and Puerto Parada in the municipality of Usulután; as well as the canton of Metalío in Sonsonate department (western El Salvador) and the DREF Final Report El Salvador: Floods Photo: Area affected by Hurricane Michael",
                             ],
     "startYear" : 2018,
     "startMonth" : 10,
     "startDay" : 6,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Tropical Storm Michael",
    },

]

In [78]:
i=12
print(f"{reports_to_label_luca.iloc[i].appealCode}: {reports_to_label_luca.iloc[i].date}")
reports_to_label_luca.iloc[i].nathaz_text

MDRYE011: 2023-05-29 00:00:00


["SITUATION ANALYSIS Description of the disaster Heavy rains in Sana'a governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.",
 'Three people died and two were injured due to the heavy rain.',
 "In Sa'adah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.",
 "Yemen's annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunderstorms starting in May 2022.",
 'According to the Food and Agriculture Organization of the United Nations (FAO) Agrometeorological Early Warning Bulletin1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.',
 'Heavy rains and flooding caused significant damage in Yemen from May to mid-September 2022, resulting in the lo

In [79]:
labelled_reports_dict["MDRYE011"]=[
    {"reportDate": "2023-05-29",
     "hazardType": "Flood",
     "hazardSubtypes" : ["flash flood"],
     "country" : "Yemen",
     "region" : ["Sana'a governorate",
                 "Sa'adah governorate",
                 "north of Ibb",
                 "central Hadramawt",
                 "Sanaa"],
     "city" : ["Gaps Ad Dali'",
               "Al Bayda",
               "Al Hodeidah",
               "Al Jawf",
               "Al Maharah",
               "Al Mahwit",
               "Amran",
               "Dhamar",
               "Hadramawt",
               "Hajjah",
               "Ibb",
               "Ma'rib",
               "Sa'dah",
               "Sana'a City",
               "Shabwah"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster Heavy rains in Sana'a governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.",
                             "In Sa'adah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.",
                             "Yemen's annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunderstorms starting in May 2022.",
                             "According to the Food and Agriculture Organization of the United Nations (FAO) Agrometeorological Early Warning Bulletin1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.",
                             "Heavy rains and flooding caused significant damage in Yemen from May to mid-September 2022, resulting in the loss of lives, destruction of property and livelihoods, and damage to infrastructure.",
                             "1 Agrometeorological Early Warning Bulletin (18-31 July 2022) [EN/AR] - Yemen | ReliefWeb Final Report Yemen: Sanaa Floods Internal Internal Internal Heavy rains and flooding continued across Yemen into the third week of August 2022.",
                             "©YRCS Internal Internal Internal Governorate Total Affected HHs since start of 2022 floods Total HHs Reached by YRCS since start of the 2022 flood Gaps Ad Dali' 647 647 0 Al Bayda 839 448 391 Al Hodeidah 1031 634 397 Al Jawf 4291 250 4041 Al Maharah 53 53 0 Al Mahwit 162 79 83 Amran 2688 497 2191 Dhamar 520 69 451 Hadramawt 1112 112 1000 Hajjah 1936 1031 905 Ibb 719 210 509 Ma'rib 23731 1750 21981 Sa'dah 229 170 59 Sana'a City 783 555 228 Sana'a Governorate 1121 407 714 Shabwah 392 1 391 Total 40,254 6,913 33,341 Of the total number of people affected, an estimated 17,000 across affected IDPs sites have suffered total damages to tents and other belongings.",
                            ],
     "startYear" : 2022,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : 2022,
     "endMonth" : 9,
     "endDay" : 15,
     "hazardName" : None,
    },
    {"reportDate": "2023-05-29",
     "hazardType": "Storm",
     "hazardSubtypes" : None,
     "country" : "Yemen",
     "region" : ["Sana'a governorate",
                 "Sa'adah governorate",
                 "north of Ibb",
                 "central Hadramawt",
                 "Sanaa"],
     "city" : ["Gaps Ad Dali'",
               "Al Bayda",
               "Al Hodeidah",
               "Al Jawf",
               "Al Maharah",
               "Al Mahwit",
               "Amran",
               "Dhamar",
               "Hadramawt",
               "Hajjah",
               "Ibb",
               "Ma'rib",
               "Sa'dah",
               "Sana'a City",
               "Shabwah"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster Heavy rains in Sana'a governorate caused extensive damage to public infrastructure, shelters for displaced people, and other private property.",
                             "In Sa'adah Governorate, 299 families were affected, and approximately 50 of them were affected by heavy rains in various districts.",
                             "Yemen's annual rainy season starts in May and normally goes until August or September, but in 2022, Yemen witnessed heavier than normal rains, ranging in intensity, accompanied by thunderstorms starting in May 2022.",
                             "According to the Food and Agriculture Organization of the United Nations (FAO) Agrometeorological Early Warning Bulletin1, the forecasts covering until July 31 favour the formation of heavy rains, especially affecting areas to the north of Ibb and central Hadramawt.",
                             "Heavy rains and flooding caused significant damage in Yemen from May to mid-September 2022, resulting in the loss of lives, destruction of property and livelihoods, and damage to infrastructure.",
                             "1 Agrometeorological Early Warning Bulletin (18-31 July 2022) [EN/AR] - Yemen | ReliefWeb Final Report Yemen: Sanaa Floods Internal Internal Internal Heavy rains and flooding continued across Yemen into the third week of August 2022.",
                             "©YRCS Internal Internal Internal Governorate Total Affected HHs since start of 2022 floods Total HHs Reached by YRCS since start of the 2022 flood Gaps Ad Dali' 647 647 0 Al Bayda 839 448 391 Al Hodeidah 1031 634 397 Al Jawf 4291 250 4041 Al Maharah 53 53 0 Al Mahwit 162 79 83 Amran 2688 497 2191 Dhamar 520 69 451 Hadramawt 1112 112 1000 Hajjah 1936 1031 905 Ibb 719 210 509 Ma'rib 23731 1750 21981 Sa'dah 229 170 59 Sana'a City 783 555 228 Sana'a Governorate 1121 407 714 Shabwah 392 1 391 Total 40,254 6,913 33,341 Of the total number of people affected, an estimated 17,000 across affected IDPs sites have suffered total damages to tents and other belongings.",
                            ],
     "startYear" : 2022,
     "startMonth" : 5,
     "startDay" : None,
     "endYear" : 2022,
     "endMonth" : 9,
     "endDay" : 15,
     "hazardName" : None,
    },
]

In [83]:
#convert to dataframe
del labelled_reports_dict["appealCode"]
df_list = []
for k,v in labelled_reports_dict.items():
    df = pd.DataFrame(v)
    df['appealCode'] = k
    df_list.append(df)
df_all = pd.concat(df_list)
df_all.reset_index(inplace=True, drop=True)

/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_58090/3402577584.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(df_list)


In [84]:
df_all.appealCode.unique()

array(['MDRBD015', 'MDRBD022', 'MDRBZ006', 'MDRCN006', 'MDRGN015',
       'MDRIQ014', 'MDRLA009', 'MDRMG020', 'MDRMY003', 'MDRNI012',
       'MDRS2001', 'MDRSV012', 'MDRYE011'], dtype=object)

In [85]:
#save
fn = "labelled_reports_luca.csv"
df_all.to_csv(DATA_LABELLED+fn)